### Generation: Generate a response


###### Setup API Key som miljøvariabel ######


In [1]:
%load_ext dotenv
%dotenv ../.env

###### Import ######

In [2]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

###### Opret Chroma Vectorstore & Embedding model ######

In [3]:
vectorstore = Chroma(persist_directory = "./chroma_db",
                     embedding_function = OpenAIEmbeddings(model='text-embedding-ada-002'))

In [4]:
len(vectorstore.get()['documents'])

22

###### Søg i Vectorstore med retriver ######


In [5]:
retriever = vectorstore.as_retriever(search_type = 'mmr',
                                     search_kwargs = {'k':3,
                                                      'lambda_mult':0.7})

###### Prompt template ######

In [6]:
TEMPLATE = '''
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture this context is taken from in the format:
Resources: *Lecture Title*
where *Lecture Title* should be substituted with the title of all resource lectures.
'''

prompt_template = PromptTemplate.from_template(TEMPLATE)

###### Chat model ######

In [8]:
chat = ChatOpenAI(model='gpt-4',
                  seed= 365,
                  max_tokens=250)

###### Definer brugerspørgsmål ######

In [9]:
question = "What software do data scientists use?"

###### Opbyg retieval chain med stuffing dokumenter ######

In [13]:
# Start med at lave en dictonary med questions og keys
chain = ({'context': retriever,
         'question': RunnablePassthrough()}
         | prompt_template
         | chat
         | StrOutputParser())

In [14]:
chain.invoke(question)

'Data scientists use several software tools and programming languages. The two most popular tools are R and Python, which are lauded for their adaptability and their integration within multiple data and data science software platforms. These tools are not just suitable for mathematical and statistical computations but can solve a wide variety of business and data-related problems. Another crucial software framework used in data science is Hadoop, designed particularly to handle the complexity and computational intensity of big data. Moreover, Power BI, SaS, Qlik, and Tableau are top-notch examples of software designed for business intelligence visualizations.\n\nResources: Programming Languages & Software Employed in Data Science - All the Tools You Need'